# Experiments — Electricity Load Forecasting


## Imports & Setup

In [ ]:
import sys
sys.path.append("..") 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

from src.experiments import config, data, features, cv, hpo
from src.experiments.mlflow_utils import log_cv_run

config.init_mlflow()


## Load Data & Split

In [ ]:
df = data.load_raw_data("../data/interim/df_core_features.parquet")
df = features.add_base_features(df)

train, test, TRAIN_END, TEST_START = data.chronological_split(df, purge_days=config.PURGE_DAYS)
train, test = features.attach_holiday_names(train, test)

print(f"Train: {len(train):,} rows | {train['timestamp'].min()} -> {train['timestamp'].max()}")
print(f"Test:  {len(test):,} rows | {test['timestamp'].min()} -> {test['timestamp'].max()}")


## Baseline: Constant & Persistence

In [ ]:
baseline_oof = cv.run_naive_baselines(train, config.FOLD_BOUNDARIES, config.TARGET)
for name, oof in baseline_oof.items():
    m = cv.compute_oof_metrics(train, oof, config.TARGET)
    print(f"{name} OOF RMSE: {m['rmse']:.2f}")


## Linear Regression (raw features)

In [ ]:
result_lr = cv.run_raw_cv(
    train, config.FOLD_BOUNDARIES, config.TARGET, config.FEATURES,
    model_builder=LinearRegression, scale_X=True,
)
metrics_lr = cv.compute_oof_metrics(train, result_lr["oof"], config.TARGET)
print(f"Linear Model, CV OOF RMSE: {metrics_lr['rmse']:.2f}  MAPE: {metrics_lr['mape']:.2f}%")

log_cv_run(
    "Baseline_Linear_Regression",
    params={"model_type": "LinearRegression", "features": "raw_features", "cv_strategy": "5-fold-expanding"},
    metrics={"cv_oof_rmse": metrics_lr["rmse"], "cv_oof_mape": metrics_lr["mape"]},
)


## XGBoost (Baseline, raw features)

In [ ]:
xgb_baseline_params = dict(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)

result_xgb_base = cv.run_raw_cv(
    train, config.FOLD_BOUNDARIES, config.TARGET, config.FEATURES,
    model_builder=lambda: xgb.XGBRegressor(**xgb_baseline_params), scale_X=True,
)
metrics_xgb_base = cv.compute_oof_metrics(train, result_xgb_base["oof"], config.TARGET)
print(f"XGBoost Baseline, CV OOF RMSE: {metrics_xgb_base['rmse']:.2f}  MAPE: {metrics_xgb_base['mape']:.2f}%")

log_cv_run(
    "Baseline_XGBoost_Raw",
    params={"model_type": "XGBoost", "features": "raw_features", "cv_strategy": "5-fold-expanding", **xgb_baseline_params},
    metrics={"cv_oof_rmse": metrics_xgb_base["rmse"], "cv_oof_mape": metrics_xgb_base["mape"]},
)


## Trend + XGBoost-on-Residual (v1)
Raw features, no derived flags — `fold_feature_fn=None`.

In [ ]:
default_gbm_params = dict(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)

result_v1 = cv.run_trend_plus_cv(
    train, config.FOLD_BOUNDARIES, config.TARGET, config.FEATURES,
    model_builder=lambda: xgb.XGBRegressor(**default_gbm_params),
    fold_feature_fn=None,
)
metrics_v1 = cv.compute_oof_metrics(train, result_v1["oof"], config.TARGET)
print(f"Trend + XGB (v1), CV OOF RMSE: {metrics_v1['rmse']:.2f}  MAPE: {metrics_v1['mape']:.2f}%")

log_cv_run(
    "Trend_XGB_v1",
    params={"model_type": "Trend+XGBoost", "experiment_version": "v1", "features": "raw_features + trend_idx",
            "cv_strategy": "5-fold-expanding", **default_gbm_params},
    metrics={"cv_oof_rmse": metrics_v1["rmse"], "cv_oof_mape": metrics_v1["mape"]},
)


## Trend + XGBoost + Extreme-Event Features (v2)
Same extreme-event fold features as v3 below — only the *feature set passed to the model* differs (`FEATURES_V2` excludes `temp_change_vs_lag24` / `is_high_precip_event`).

In [ ]:
result_v2 = cv.run_trend_plus_cv(
    train, config.FOLD_BOUNDARIES, config.TARGET, config.FEATURES_V2,
    model_builder=lambda: xgb.XGBRegressor(**default_gbm_params),
    fold_feature_fn=features.extreme_event_fold_features,
)
metrics_v2 = cv.compute_oof_metrics(train, result_v2["oof"], config.TARGET)
print(f"Trend + XGB + extreme events (v2), CV OOF RMSE: {metrics_v2['rmse']:.2f}  (v1: {metrics_v1['rmse']:.2f})")

log_cv_run(
    "Trend_XGB_v2",
    params={"model_type": "Trend+XGBoost", "experiment_version": "v2",
            "features": "FEATURES_V2 (extreme events + holiday interaction)",
            "cv_strategy": "5-fold-expanding", "extreme_heat_quantile": 0.95, "extreme_cold_quantile": 0.05,
            **default_gbm_params},
    metrics={"cv_oof_rmse": metrics_v2["rmse"], "cv_oof_mape": metrics_v2["mape"]},
)


## Model Sweep on v3 Features (XGBoost / LightGBM / CatBoost / KNN)
This replaces four separate copy-pasted cells — same CV runner, only the model builder changes.

In [ ]:
v3_fold_features = features.extreme_event_fold_features

model_configs = {
    "xgb_v3":      dict(builder=lambda: xgb.XGBRegressor(**default_gbm_params), scale_X=False,
                         mlflow_name="Trend_XGB_v3", model_label="Trend+XGBoost",
                         hp=default_gbm_params),
    "lgbm_v3":     dict(builder=lambda: lgb.LGBMRegressor(**default_gbm_params, verbosity=-1), scale_X=False,
                         mlflow_name="Trend_LightGBM_v3", model_label="Trend+LightGBM",
                         hp=default_gbm_params),
    "catboost_v3": dict(builder=lambda: cb.CatBoostRegressor(
                             iterations=500, learning_rate=0.05, depth=6,
                             random_state=42, thread_count=-1, verbose=0), scale_X=False,
                         mlflow_name="Trend_CatBoost_v3", model_label="Trend+CatBoost",
                         hp=dict(iterations=500, learning_rate=0.05, depth=6)),
    "knn_v3":      dict(builder=lambda: KNeighborsRegressor(n_neighbors=15, weights="distance", n_jobs=-1), scale_X=True,
                         mlflow_name="Trend_KNN_v3", model_label="Trend+KNN",
                         hp=dict(n_neighbors=15, weights="distance")),
}

v3_results, v3_metrics = {}, {}
for name, cfg in model_configs.items():
    print(f"--- {name} ---")
    result = cv.run_trend_plus_cv(
        train, config.FOLD_BOUNDARIES, config.TARGET, config.FEATURES_V3,
        model_builder=cfg["builder"], fold_feature_fn=v3_fold_features, scale_X=cfg["scale_X"],
    )
    m = cv.compute_oof_metrics(train, result["oof"], config.TARGET)
    v3_results[name] = result
    v3_metrics[name] = m
    print(f"{cfg['model_label']} (v3), CV OOF RMSE: {m['rmse']:.2f}  MAPE: {m['mape']:.2f}%\n")

    log_cv_run(
        cfg["mlflow_name"],
        params={"model_type": cfg["model_label"], "experiment_version": "v3",
                "features": "FEATURES_V3", "cv_strategy": "5-fold-expanding", **cfg["hp"]},
        metrics={"cv_oof_rmse": m["rmse"], "cv_oof_mape": m["mape"]},
    )


## Model Comparison — RMSE / MAPE / Peak-MAPE

In [ ]:
oof_dict_v3 = {name: v3_results[name]["oof"] for name in model_configs}
oof_dict_v3["lr"] = result_lr["oof"]  # note: lr ran on raw FEATURES, not v3 — included for reference only

model_rows = []
for name, oof in oof_dict_v3.items():
    valid = ~np.isnan(oof)
    y_true, y_pred = train.loc[valid, config.TARGET], oof[valid]
    m = cv.compute_metrics(y_true.to_numpy(), y_pred)
    model_rows.append({"model": name, **m})

model_comparison = pd.DataFrame(model_rows).sort_values("peak_mape").reset_index(drop=True)
model_comparison


## Residual Correlation Check

In [ ]:
common_valid = np.ones(len(train), dtype=bool)
for oof in oof_dict_v3.values():
    common_valid &= ~np.isnan(oof)

resid_df = pd.DataFrame({
    name: (train[config.TARGET].values - oof)[common_valid]
    for name, oof in oof_dict_v3.items()
})
print(f"Rows compared: {common_valid.sum():,}")
resid_df.corr()


## Hill-Climbing Ensemble

In [ ]:
y_true_full = train[config.TARGET].values
ensemble_preds, selected, ensemble_weights, ensemble_history, solo_rmse = cv.hill_climb_ensemble(
    oof_dict_v3, y_true_full, common_valid
)

print("Solo RMSE (on common_valid rows):")
for name, rmse in sorted(solo_rmse.items(), key=lambda x: x[1]):
    print(f"  {name}: {rmse:.2f}")

print(f"\nHill-climb ensemble OOF RMSE: {ensemble_history[-1]:.2f}")
print("\nModel weights (by selection frequency):")
print(ensemble_weights)


## Hyperparameter Tuning — XGBoost / LightGBM / CatBoost (Optuna)
Each `run_study` call replaces a separate hand-written objective + CV loop — same shared `oof_rmse_for_model` harness underneath for all three.

In [ ]:
studies = {}
for model_name in ("xgb", "lgbm", "catboost"):
    baseline_rmse = v3_metrics[f"{model_name}_v3"]["rmse"]
    study = hpo.run_study(
        model_name, train, config.FOLD_BOUNDARIES, config.TARGET, config.FEATURES_V3,
        fold_feature_fn=v3_fold_features, n_trials=40,
    )
    studies[model_name] = study
    print(f"Best {model_name} OOF RMSE: {study.best_value:.2f}  (default-params baseline: {baseline_rmse:.2f})")
    print(study.best_params)


## Tuned Model Comparison

In [ ]:
tuned_comparison = pd.DataFrame([
    {"model": name, "tuned_oof_rmse": study.best_value}
    for name, study in studies.items()
]).sort_values("tuned_oof_rmse").reset_index(drop=True)
tuned_comparison


## Target-Encoding Check — Holiday Identity

In [ ]:
CHAMPION_NAME = tuned_comparison.iloc[0]["model"]
CHAMPION_OOF_RMSE = tuned_comparison.iloc[0]["tuned_oof_rmse"]
CHAMPION_PARAMS = studies[CHAMPION_NAME].best_params

champion_fold_features = features.compose(features.extreme_event_fold_features, features.holiday_freq_fold_features)
champion_model_fn = lambda: hpo.MODEL_BUILDERS[CHAMPION_NAME](CHAMPION_PARAMS)

holiday_oof_rmse = hpo.oof_rmse_for_model(
    train, config.FOLD_BOUNDARIES, config.TARGET, config.FEATURES_V3 + ["holiday_freq"],
    champion_model_fn, champion_fold_features,
)

print(f"Champion ({CHAMPION_NAME}) OOF RMSE without holiday_freq: {CHAMPION_OOF_RMSE:.2f}")
print(f"Champion ({CHAMPION_NAME}) OOF RMSE with holiday_freq:    {holiday_oof_rmse:.2f}")

USE_HOLIDAY_FEATURE = holiday_oof_rmse < CHAMPION_OOF_RMSE
print(f"\nUse holiday_freq in final model: {USE_HOLIDAY_FEATURE}")


## Champion Selection & Final Model Fit
Same fit/apply feature functions as every CV fold above — only fit on the full `train` set this time instead of a fold slice.

In [ ]:
WINNING_FEATURES = config.FEATURES_V3 + ["holiday_freq"] if USE_HOLIDAY_FEATURE else config.FEATURES_V3

thresholds = features.fit_extreme_event_thresholds(train)
train_final = features.apply_extreme_event_features(train, thresholds)
test_final = features.apply_extreme_event_features(test, thresholds)

holiday_freq_map = None
if USE_HOLIDAY_FEATURE:
    holiday_freq_map = features.fit_holiday_freq_map(train_final)
    train_final = features.apply_holiday_freq(train_final, holiday_freq_map)
    test_final = features.apply_holiday_freq(test_final, holiday_freq_map)

X_train_final, y_train_final = train_final[WINNING_FEATURES], train_final[config.TARGET]
X_test_final, y_test_final = test_final[WINNING_FEATURES], test_final[config.TARGET]

trend_model_final = LinearRegression().fit(train_final[["trend_idx"]], y_train_final)
trend_train_final = trend_model_final.predict(train_final[["trend_idx"]])
trend_test_final = trend_model_final.predict(test_final[["trend_idx"]])

residual_target_final = y_train_final - trend_train_final

final_model = hpo.MODEL_BUILDERS[CHAMPION_NAME](CHAMPION_PARAMS)
final_model.fit(X_train_final, residual_target_final)
pred_test_final = trend_test_final + final_model.predict(X_test_final)

test_metrics = cv.compute_metrics(y_test_final.to_numpy(), pred_test_final)

print(f"Champion model: {CHAMPION_NAME}  |  Features: {'v3 + holiday_freq' if USE_HOLIDAY_FEATURE else 'v3'}")
print(f"Final test RMSE: {test_metrics['rmse']:.2f}")
print(f"Final test MAPE: {test_metrics['mape']:.3f}%")
print(f"Final test Peak-MAPE: {test_metrics['peak_mape']:.3f}%")


## MLflow Logging — Champion Model

In [ ]:
import joblib
from pathlib import Path
import mlflow
import mlflow.pyfunc


class ElectricityForecaster(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self.bundle = joblib.load(context.artifacts["model_bundle"])
        self.trend_model = self.bundle["trend_model"]
        self.residual_model = self.bundle["residual_model"]
        self.features = self.bundle["features"]

    def predict(self, context, model_input):
        df = model_input if isinstance(model_input, pd.DataFrame) else pd.DataFrame(model_input)
        trend_preds = self.trend_model.predict(df[["trend_idx"]])
        residual_preds = self.residual_model.predict(df[self.features])
        return trend_preds + residual_preds


artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
model_path = artifact_dir / "electricity_load_forecaster.pkl"

production_bundle = {
    "trend_model": trend_model_final,
    "residual_model": final_model,
    "features": WINNING_FEATURES,
    "target": config.TARGET,
    "thresholds": thresholds,
    "holiday_freq_map": holiday_freq_map,  # None if USE_HOLIDAY_FEATURE is False
    "model_family": f"trend_plus_tuned_{CHAMPION_NAME}",
    "feature_version": "v3_holiday" if USE_HOLIDAY_FEATURE else "v3",
    "train_end": str(train_final["timestamp"].max()),
    "test_start": str(test_final["timestamp"].min()),
}
joblib.dump(production_bundle, model_path)

with mlflow.start_run(run_name="production_candidate_final"):
    mlflow.log_params({
        "model_family": production_bundle["model_family"],
        "feature_version": production_bundle["feature_version"],
        **{f"{CHAMPION_NAME}_{k}": v for k, v in CHAMPION_PARAMS.items()},
        "train_rows": len(train_final),
        "test_rows": len(test_final),
        "purge_days": config.PURGE_DAYS,
        "train_end": production_bundle["train_end"],
        "test_start": production_bundle["test_start"],
    })
    mlflow.log_metrics({
        "cv_oof_rmse": float(CHAMPION_OOF_RMSE),
        "test_rmse": float(test_metrics["rmse"]),
        "test_mape_pct": float(test_metrics["mape"]),
        "test_peak_mape_pct": float(test_metrics["peak_mape"]),
    })

    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=ElectricityForecaster(),
        artifacts={"model_bundle": str(model_path)},
        registered_model_name="Electricity-Load-Forecaster",
    )

print("Successfully logged params, metrics, and registered the production candidate model!")
